[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/33.%20The%20Labor%20Shift%20Scheduling%20Problem/P33-Tier-8_executed.ipynb)

---


# 33. The Labor Shift Scheduling Problem

## Tier 8: The Value-Aligned and Ethical Framework

### Goal

This notebook demonstrates a transparent, rule-based greedy procedure for assigning healthcare staff to shifts while considering operational coverage, workload distribution, and selected worker-protection limits. It also illustrates how fairness indicators, bias screening, feedback records, and audit records can be organized around a scheduling process.

### Inputs and outputs

The example uses a synthetic workforce, seven days, three shifts per day, employee availability, and a demand count for each day-shift pair. The scheduler produces a mapping from each employee and day to at most one shift, together with coverage and ethical-compliance indicators. Because the procedure is greedy, it may leave some demand uncovered; coverage is measured and reported rather than assumed.

### Method and mathematical representation

The method processes day-shift requirements in the order supplied. For each requirement, it filters out unavailable staff and staff who would violate the implemented assignment limits. It then scores each eligible employee and selects the highest-scoring candidates until demand is met or no eligible candidates remain. This is a sequential greedy heuristic, not a proof of global optimality.

Let $E$ be the set of employees, $T$ the ordered set of days, and $S$ the set of shifts. Let $d_{ts}$ be demand for shift $s$ on day $t$. For each employee $e$, day $t$, and shift $s$, let $x_{ets}=1$ if employee $e$ is assigned to that shift and $0$ otherwise. The schedule representation permits at most one shift per employee per day:

$$\sum_{s\in S} x_{ets} \leq 1 \qquad (e\in E,\ t\in T).$$

The greedy procedure attempts to cover demand without exceeding the number of eligible candidates:

$$\sum_{e\in E} x_{ets} \leq d_{ts} \qquad (t\in T,\ s\in S).$$

Eligibility incorporates availability, a maximum of five assignments in the example week, a limit on consecutive assigned days, and a weekly night-shift limit derived from the framework's monthly limit. These are local selection rules: once a candidate is selected, the schedule is updated before the next day-shift requirement is processed. The code does not optimize a single global objective or backtrack to revise earlier assignments.

For an eligible employee $e$, the selection score is

$$q_e = 0.4\,r_e + 0.6\,u_e,$$

where $r_e$ is a randomly generated operational score and $u_e$ is a score based on current assignment count, night-shift count, and consecutive-day considerations. The coefficients are the scheduler's stated weights. A random component makes runs stochastic; no random seed is set in this notebook, so exact schedules and simulated feedback are not reproducible across runs.

### Scope and modeling assumptions

- Employees, availability, demographics, and feedback are synthetic. No real worker, patient, or organizational data are used.
- Shifts are represented by names, not start and end times. Minimum rest hours therefore cannot be calculated or enforced.
- The code implements selected assignment limits, but the fairness threshold is a reporting indicator rather than a hard constraint. The model does not enforce every item listed in the ethical framework.
- The random operational score is not a measured performance or qualification score. Demographic attributes are not used to rank employees.
- Department and experience-group disparities are screening indicators, not proof of discrimination or causal bias. A real assessment needs suitable data, context, stakeholder review, and legally appropriate methods.
- Compliance scores are limited indicators derived from this demonstration. They do not establish regulatory compliance, improved retention, reduced burnout, or stakeholder trust.

### Ethical principles used in the example

The framework organizes evaluation around fairness, transparency, well-being, accountability, and sustainability. These categories help structure an audit, but a numerical score cannot by itself determine whether a schedule is ethically acceptable. In particular, this example has no recorded explanation for each assignment and does not collect real stakeholder feedback.

In [ ]:
# Import the packages used for scheduling, tabulation, and visualization.
import random
import time
from collections import defaultdict
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
class EthicalPrinciples:
    """Store the evaluation categories and limits used in the example."""

    def __init__(self):
        self.principles = {
            "fairness": {"description": "Distribution of assigned hours", "weight": 0.25},
            "transparency": {"description": "Visibility of assignment records", "weight": 0.20},
            "well_being": {"description": "Selected workload protections", "weight": 0.25},
            "accountability": {"description": "Availability of audit records", "weight": 0.15},
            "sustainability": {"description": "Coverage and workload feasibility", "weight": 0.15},
        }
        self.constraints = {
            "max_consecutive_shifts": 5,
            "min_rest_hours": 11,
            "max_weekly_hours": 48,
            "min_days_off_per_week": 2,
            "max_night_shifts_per_month": 8,
            "fair_distribution_threshold": 0.1,
        }

    def evaluate_ethical_compliance(self, schedule, employees):
        """Calculate indicators from the schedule and identify unmeasured items."""
        employee_hours = {employee.id: 0 for employee in employees}
        employee_days = defaultdict(list)
        employee_nights = defaultdict(int)

        for (employee_id, day), shift_name in schedule.items():
            employee_hours[employee_id] += 8
            employee_days[employee_id].append(day)
            if shift_name == "Night":
                employee_nights[employee_id] += 1

        hours = list(employee_hours.values())
        average_hours = float(np.mean(hours)) if hours else 0.0
        variance = float(np.var(hours)) if hours else 0.0
        fairness_score = (
            1.0 if average_hours == 0 else max(0.0, 1.0 - variance / average_hours**2)
        )

        days_order = [
            "Monday", "Tuesday", "Wednesday", "Thursday",
            "Friday", "Saturday", "Sunday",
        ]
        consecutive_ok = []
        for employee in employees:
            indices = sorted(days_order.index(day) for day in employee_days[employee.id])
            longest_run = 0
            current_run = 0
            previous_index = None
            for day_index in indices:
                current_run = current_run + 1 if previous_index == day_index - 1 else 1
                longest_run = max(longest_run, current_run)
                previous_index = day_index
            consecutive_ok.append(
                longest_run <= self.constraints["max_consecutive_shifts"]
            )

        weekly_hours_ok = [
            employee_hours[employee.id] <= self.constraints["max_weekly_hours"]
            for employee in employees
        ]
        weekly_night_limit = self.constraints["max_night_shifts_per_month"] // 4
        night_limit_ok = [
            employee_nights[employee.id] <= weekly_night_limit
            for employee in employees
        ]

        mean_or_one = lambda values: float(np.mean(values)) if values else 1.0
        compliance_scores = {
            "fairness": fairness_score,
            # Assignment records omit the rationale for choosing each employee.
            "transparency": 0.0,
            "well_being": mean_or_one(
                [mean_or_one(weekly_hours_ok), mean_or_one(consecutive_ok), mean_or_one(night_limit_ok)]
            ),
            # Audit capability is reported separately; this score does not certify oversight.
            "accountability": 0.0,
            "sustainability": mean_or_one(weekly_hours_ok),
        }
        overall_score = sum(
            compliance_scores[name] * self.principles[name]["weight"]
            for name in self.principles
        )
        return compliance_scores, overall_score


class EthicalScheduler:
    """Greedy scheduler that ranks eligible employees for each demand slot."""

    def __init__(self, employees, shifts, days, demand, ethical_framework):
        self.employees = employees
        self.shifts = shifts
        self.days = days
        self.demand = demand
        self.ethical_framework = ethical_framework
        self.optimization_weights = {
            "operational_efficiency": 0.4,
            "ethical_compliance": 0.6,
        }
        self.bias_detector = BiasDetector()
        self.stakeholder_system = StakeholderFeedbackSystem()
        self.audit_system = AuditSystem()
        self.coverage = {}

    def generate_ethical_schedule(self):
        """Build a schedule in day-shift order without revising earlier choices."""
        schedule = {}
        employee_assignments = {employee.id: [] for employee in self.employees}

        for day in self.days:
            for shift in self.shifts:
                required = self.demand[day][shift.name]
                available = [
                    employee for employee in self.employees
                    if employee.is_available(day, shift.name)
                    and len(employee_assignments[employee.id]) < employee.max_shifts_per_week
                    and self._within_assignment_limits(
                        employee, day, shift, employee_assignments
                    )
                ]
                selected = self._ethical_employee_selection(
                    available, required, day, shift, employee_assignments
                )

                for employee in selected:
                    schedule[(employee.id, day)] = shift.name
                    employee_assignments[employee.id].append((day, shift.name))

                self.coverage[(day, shift.name)] = len(selected)

        compliance_scores, overall_score = (
            self.ethical_framework.evaluate_ethical_compliance(schedule, self.employees)
        )
        self._generate_ethical_report(schedule, compliance_scores, overall_score)
        return schedule, compliance_scores, overall_score

    def _within_assignment_limits(self, employee, day, shift, assignments):
        """Check the local workload limits before a candidate is ranked."""
        current = assignments[employee.id]
        if any(assigned_day == day for assigned_day, _ in current):
            return False

        constraints = self.ethical_framework.constraints
        if shift.name == "Night":
            night_limit = constraints["max_night_shifts_per_month"] // 4
            if sum(name == "Night" for _, name in current) >= night_limit:
                return False

        day_order = {name: index for index, name in enumerate(self.days)}
        assigned_indices = sorted(day_order[assigned_day] for assigned_day, _ in current)
        assigned_indices.append(day_order[day])
        assigned_indices.sort()
        longest_run = 0
        current_run = 0
        previous_index = None
        for index in assigned_indices:
            current_run = current_run + 1 if previous_index == index - 1 else 1
            longest_run = max(longest_run, current_run)
            previous_index = index
        return longest_run <= constraints["max_consecutive_shifts"]

    def _ethical_employee_selection(self, available_employees, required, day, shift, assignments):
        """Rank eligible employees using the stated weighted score."""
        employee_scores = []
        for employee in available_employees:
            operational_score = random.uniform(0.6, 0.9)
            ethical_score = self._calculate_ethical_score(
                employee, day, shift, assignments
            )
            combined_score = (
                operational_score * self.optimization_weights["operational_efficiency"]
                + ethical_score * self.optimization_weights["ethical_compliance"]
            )
            employee_scores.append((employee, combined_score))

        employee_scores.sort(key=lambda item: item[1], reverse=True)
        return [employee for employee, _ in employee_scores[:required]]

    def _calculate_ethical_score(self, employee, day, shift, assignments):
        """Reward lower current workload and discourage repeated night work."""
        score = 1.0
        counts = [len(assignments[item.id]) for item in self.employees]
        average_assignments = float(np.mean(counts)) if counts else 0.0
        current_count = len(assignments[employee.id])
        if average_assignments > 0 and current_count > average_assignments * 1.2:
            score *= 0.7

        if shift.name == "Night":
            previous_nights = sum(
                name == "Night" for _, name in assignments[employee.id]
            )
            if previous_nights >= 1:
                score *= 0.8

        if assignments[employee.id]:
            day_order = {name: index for index, name in enumerate(self.days)}
            current_index = day_order[day]
            if any(
                abs(day_order[assigned_day] - current_index) == 1
                for assigned_day, _ in assignments[employee.id]
            ):
                score *= 0.9
        return score

    def _generate_ethical_report(self, schedule, compliance_scores, overall_score):
        """Report indicators and record the schedule for audit review."""
        print("\nETHICAL INDICATOR REPORT")
        print(f"Weighted indicator score: {overall_score:.2%}")
        for principle, score in compliance_scores.items():
            print(f"  {principle.replace('_', ' ').title()}: {score:.2%}")
        self.audit_system.record_schedule_audit(
            schedule, compliance_scores, overall_score
        )


class BiasDetector:
    """Screen assignment rates for disparities across selected employee groups."""

    def detect_bias(self, schedule, employees):
        """Return department and experience-group disparity indicators."""
        assignment_counts = {employee.id: 0 for employee in employees}
        for employee_id, _ in schedule:
            assignment_counts[employee_id] += 1

        department_rates = defaultdict(list)
        for employee in employees:
            department_rates[employee.department].append(assignment_counts[employee.id])
        department_means = [float(np.mean(values)) for values in department_rates.values()]
        department_bias = self._normalized_gap(department_means)

        experience_groups = {"0-5 years": [], "More than 5 years": []}
        for employee in employees:
            group = "0-5 years" if employee.experience_years <= 5 else "More than 5 years"
            experience_groups[group].append(assignment_counts[employee.id])
        experience_means = [
            float(np.mean(values)) for values in experience_groups.values() if values
        ]
        experience_bias = self._normalized_gap(experience_means)
        analysis = {
            "department_bias": department_bias,
            "experience_bias": experience_bias,
        }
        return analysis, float(np.mean(list(analysis.values())))

    @staticmethod
    def _normalized_gap(group_means):
        if len(group_means) < 2:
            return 0.0
        scale = max(group_means)
        return 0.0 if scale == 0 else min(1.0, (max(group_means) - min(group_means)) / scale)

    def generate_mitigation_recommendations(self, bias_analysis):
        """Suggest a review when a screening indicator exceeds the chosen threshold."""
        recommendations = []
        for bias_type, score in bias_analysis.items():
            if score > 0.2:
                group_name = bias_type.replace("_bias", "").replace("_", " ")
                recommendations.append(
                    f"Review assignment rates and context for the {group_name} groups."
                )
        return recommendations


class StakeholderFeedbackSystem:
    """Store and summarize explicitly synthetic feedback records."""

    def __init__(self):
        self.stakeholders = ["employees", "managers", "patients", "administrators"]
        self.feedback_data = defaultdict(list)

    def collect_feedback(self, stakeholder_type, feedback_score, comments=None):
        if stakeholder_type not in self.stakeholders:
            raise ValueError("Unknown stakeholder type")
        if not 1 <= feedback_score <= 5:
            raise ValueError("Feedback scores must be between 1 and 5")
        self.feedback_data[stakeholder_type].append(
            {"timestamp": datetime.now(), "score": feedback_score, "comments": comments}
        )

    def analyze_feedback(self):
        analysis = {}
        for stakeholder in self.stakeholders:
            entries = self.feedback_data[stakeholder]
            scores = [entry["score"] for entry in entries]
            trend = "no_data"
            if len(scores) > 1:
                trend = "improving" if scores[-1] > scores[0] else "declining" if scores[-1] < scores[0] else "stable"
            analysis[stakeholder] = {
                "average_score": float(np.mean(scores)) if scores else 0.0,
                "count": len(scores),
                "trend": trend,
            }
        return analysis

    def generate_feedback_report(self):
        print("\nSYNTHETIC STAKEHOLDER FEEDBACK SUMMARY")
        for stakeholder, data in self.analyze_feedback().items():
            if data["count"]:
                print(
                    f"{stakeholder.title()}: {data['average_score']:.2f}/5 "
                    f"from {data['count']} simulated responses ({data['trend']})"
                )
            else:
                print(f"{stakeholder.title()}: no feedback recorded")


class AuditSystem:
    """Keep an in-memory record of schedule indicator reports."""

    def __init__(self):
        self.audit_log = []
        self.compliance_history = []

    def record_schedule_audit(self, schedule, compliance_scores, overall_score):
        entry = {
            "timestamp": datetime.now(),
            "assignment_count": len(schedule),
            "compliance_scores": dict(compliance_scores),
            "overall_score": overall_score,
        }
        self.audit_log.append(entry)
        self.compliance_history.append(overall_score)

    def generate_audit_report(self):
        print("\nAUDIT RECORD SUMMARY")
        if not self.audit_log:
            print("No audit records are available.")
            return
        for entry in self.audit_log[-5:]:
            print(
                f"{entry['timestamp'].isoformat(timespec='seconds')}: "
                f"{entry['assignment_count']} assignments, "
                f"indicator score {entry['overall_score']:.2%}"
            )
        print(f"Records stored in this session: {len(self.audit_log)}")

In [ ]:
class Employee:
    """Synthetic employee record with availability and preference attributes."""

    def __init__(self, id, name, department=None, experience_years=0, demographics=None):
        self.id = id
        self.name = name
        self.department = department
        self.experience_years = experience_years
        self.demographics = demographics or {}
        self.preferences = self._generate_preferences()
        self.availability_patterns = self._generate_availability()
        self.max_shifts_per_week = 5
        self.assigned_shifts = 0
        self.stress_level = random.uniform(1, 3)
        self.satisfaction_score = random.uniform(3, 5)
        self.burnout_risk = random.uniform(0, 0.3)
        self.base_hourly_rate = 25 + experience_years * 2
        self.performance_rating = 4.0

    def _generate_preferences(self):
        preferences = {}
        for shift_name in ["Morning", "Afternoon", "Night"]:
            preference = random.choice([-1, 0, 0, 1])
            if preference:
                preferences[shift_name] = preference
        return preferences

    def _generate_availability(self):
        days = [
            "Monday", "Tuesday", "Wednesday", "Thursday",
            "Friday", "Saturday", "Sunday",
        ]
        shifts = ["Morning", "Afternoon", "Night"]
        availability = {}
        for day in days:
            for shift_name in shifts:
                probability = 0.85 if day in days[:5] else 0.7
                preference = self.preferences.get(shift_name, 0)
                if preference > 0:
                    probability = min(1.0, probability + 0.15)
                elif preference < 0:
                    probability = max(0.0, probability - 0.25)
                availability[(day, shift_name)] = random.random() < probability
        return availability

    def is_available(self, day, shift):
        return self.availability_patterns.get((day, shift), False)

    def get_cost(self, shift, day):
        cost = self.base_hourly_rate * 8
        if shift == "Night":
            cost *= 1.5
        elif shift == "Afternoon":
            cost *= 1.2
        if day in ["Saturday", "Sunday"]:
            cost *= 1.3
        return cost

    def update_well_being_metrics(self, schedule_assignment):
        """Apply illustrative changes from supplied assignment summary data."""
        if schedule_assignment.get("night_shifts", 0) > 2:
            self.stress_level = min(5.0, self.stress_level + 0.1)
            self.burnout_risk = min(1.0, self.burnout_risk + 0.05)
        if schedule_assignment.get("consecutive_days", 0) > 3:
            self.satisfaction_score = max(1.0, self.satisfaction_score - 0.2)
        if schedule_assignment.get("preferred_shifts", 0) > 0:
            self.satisfaction_score = min(5.0, self.satisfaction_score + 0.1)
            self.stress_level = max(1.0, self.stress_level - 0.05)


class Shift:
    """Shift label used by the day-shift demand table."""

    def __init__(self, id, name, required_skills=None):
        self.id = id
        self.name = name
        self.required_skills = required_skills or ["basic"]

In [ ]:
def create_ethical_scenario():
    """Create a synthetic healthcare scheduling instance."""
    employees = []
    departments = [
        "Emergency", "ICU", "Surgery", "Pediatrics",
        "General Medicine", "Radiology",
    ]
    for index in range(50):
        department = departments[index % len(departments)]
        experience = random.randint(0, 20)
        demographics = {
            "age_group": random.choice(["25-34", "35-44", "45-54", "55+"]),
            "gender": random.choice(["male", "female", "non-binary"]),
            "ethnicity": random.choice(["white", "black", "hispanic", "asian", "other"]),
        }
        employees.append(
            Employee(
                id=index + 1,
                name=f"Staff_{index + 1:03d}",
                department=department,
                experience_years=experience,
                demographics=demographics,
            )
        )

    shifts = [Shift(1, "Morning"), Shift(2, "Afternoon"), Shift(3, "Night")]
    days = [
        "Monday", "Tuesday", "Wednesday", "Thursday",
        "Friday", "Saturday", "Sunday",
    ]
    demand = {
        "Monday": {"Morning": 15, "Afternoon": 18, "Night": 8},
        "Tuesday": {"Morning": 14, "Afternoon": 17, "Night": 8},
        "Wednesday": {"Morning": 15, "Afternoon": 18, "Night": 8},
        "Thursday": {"Morning": 14, "Afternoon": 17, "Night": 8},
        "Friday": {"Morning": 16, "Afternoon": 19, "Night": 9},
        "Saturday": {"Morning": 12, "Afternoon": 14, "Night": 7},
        "Sunday": {"Morning": 10, "Afternoon": 12, "Night": 6},
    }
    return employees, shifts, days, demand


employees, shifts, days, demand = create_ethical_scenario()
total_demand = sum(
    demand[day][shift.name] for day in days for shift in shifts
)
age_groups = [employee.demographics["age_group"] for employee in employees]
genders = [employee.demographics["gender"] for employee in employees]

print("Synthetic Healthcare Scheduling Scenario")
print(f"Employees: {len(employees)}")
print(f"Departments: {len({employee.department for employee in employees})}")
print(f"Shifts per day: {len(shifts)}")
print(f"Days: {len(days)}")
print(f"Demand slots: {total_demand}")
print("Demographic counts are descriptive synthetic data, not fairness evidence.")
print("Age groups:", dict(zip(*np.unique(age_groups, return_counts=True))))
print("Gender categories:", dict(zip(*np.unique(genders, return_counts=True))))

In [ ]:
ethical_framework = EthicalPrinciples()
ethical_scheduler = EthicalScheduler(
    employees, shifts, days, demand, ethical_framework
)

start_time = time.time()
ethical_schedule, compliance_scores, overall_score = (
    ethical_scheduler.generate_ethical_schedule()
)
execution_time = time.time() - start_time

assigned_slots = sum(ethical_scheduler.coverage.values())
coverage_rate = assigned_slots / total_demand if total_demand else 1.0
print(f"\nGreedy scheduling runtime: {execution_time:.3f} seconds")
print(f"Assigned demand slots: {assigned_slots} of {total_demand}")
print(f"Demand coverage: {coverage_rate:.1%}")
print("The weighted indicator score is not a certification of ethical compliance.")

In [ ]:
bias_analysis, overall_bias = ethical_scheduler.bias_detector.detect_bias(
    ethical_schedule, employees
)
print("\nASSIGNMENT-RATE DISPARITY SCREEN")
print(f"Mean of displayed disparity indicators: {overall_bias:.3f}")
print("Lower values indicate smaller observed gaps in this synthetic schedule.")
for bias_type, score in bias_analysis.items():
    print(f"  {bias_type.replace('_', ' ').title()}: {score:.3f}")

recommendations = ethical_scheduler.bias_detector.generate_mitigation_recommendations(
    bias_analysis
)
if recommendations:
    print("Review prompts:")
    for number, recommendation in enumerate(recommendations, start=1):
        print(f"  {number}. {recommendation}")
else:
    print("No indicator exceeded the selected review threshold.")
print("These descriptive gaps do not establish the presence or absence of bias.")

In [ ]:
stakeholder_system = ethical_scheduler.stakeholder_system
print("\nGenerating synthetic feedback for demonstration only.")

feedback_ranges = {
    "employees": (30, 3.0, 5.0),
    "managers": (10, 3.5, 4.5),
    "patients": (15, 4.0, 4.8),
    "administrators": (5, 3.8, 4.2),
}
for stakeholder, (count, lower, upper) in feedback_ranges.items():
    for _ in range(count):
        stakeholder_system.collect_feedback(
            stakeholder,
            random.uniform(lower, upper),
            comments="Synthetic demonstration response",
        )
stakeholder_system.generate_feedback_report()
print("Simulated responses must not be interpreted as stakeholder research.")

In [ ]:
ethical_scheduler.audit_system.generate_audit_report()


def display_ethical_schedule(schedule, employees, days):
    """Display a compact schedule table for the first 15 employees."""
    displayed_employees = employees[:15]
    schedule_matrix = {
        employee.name: {day: "Off" for day in days}
        for employee in displayed_employees
    }
    employee_names = {employee.id: employee.name for employee in displayed_employees}
    for (employee_id, day), shift_name in schedule.items():
        employee_name = employee_names.get(employee_id)
        if employee_name is not None:
            schedule_matrix[employee_name][day] = shift_name[:3]

    dataframe = pd.DataFrame.from_dict(schedule_matrix, orient="index")
    dataframe.index.name = "Employee"
    dataframe.columns.name = "Day"
    print("\nSCHEDULE SAMPLE: FIRST 15 EMPLOYEES")
    print(dataframe.to_string())
    return dataframe


ethical_schedule_df = display_ethical_schedule(
    ethical_schedule, employees, days
)

# These checks validate the schedule representation and reported coverage.
assert len(ethical_schedule) == assigned_slots
assert all(0 <= count <= demand[day][shift.name]
           for (day, shift_name), count in ethical_scheduler.coverage.items()
           for shift in shifts if shift.name == shift_name)
assert all(shift_name in {shift.name for shift in shifts}
           for shift_name in ethical_schedule.values())

In [ ]:
def create_ethical_visualizations(
    compliance_scores,
    bias_analysis,
    stakeholder_system,
    schedule_dataframe,
    employees,
    ethical_schedule,
    ethical_scheduler,
    days,
    overall_score,
):
    """Plot indicators and schedule data without inventing outcome histories."""
    figure, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.ravel()

    principle_names = [name.replace("_", " ").title() for name in compliance_scores]
    principle_values = list(compliance_scores.values())
    axes[0].bar(principle_names, principle_values, color="steelblue")
    axes[0].set_ylim(0, 1)
    axes[0].set_ylabel("Indicator score")
    axes[0].set_title("Ethical framework indicators")
    axes[0].tick_params(axis="x", rotation=45)

    bias_names = [name.replace("_", " ").title() for name in bias_analysis]
    axes[1].bar(bias_names, list(bias_analysis.values()), color="darkorange")
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel("Normalized group-rate gap")
    axes[1].set_title("Disparity screening indicators")
    axes[1].tick_params(axis="x", rotation=30)

    feedback = stakeholder_system.analyze_feedback()
    stakeholders = [name for name, data in feedback.items() if data["count"]]
    axes[2].bar(
        stakeholders,
        [feedback[name]["average_score"] for name in stakeholders],
        color="seagreen",
    )
    axes[2].set_ylim(0, 5)
    axes[2].set_ylabel("Mean simulated score (1-5)")
    axes[2].set_title("Synthetic feedback demonstration")
    axes[2].tick_params(axis="x", rotation=30)

    shift_codes = {"Off": 0, "Mor": 1, "Aft": 2, "Nig": 3}
    numeric_schedule = schedule_dataframe.replace(shift_codes).astype(float)
    image = axes[3].imshow(
        numeric_schedule.to_numpy(), cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=3
    )
    axes[3].set_xticks(range(len(days)))
    axes[3].set_xticklabels(days, rotation=45)
    axes[3].set_yticks(range(len(schedule_dataframe.index)))
    axes[3].set_yticklabels(schedule_dataframe.index)
    axes[3].set_xlabel("Day")
    axes[3].set_ylabel("Employee")
    axes[3].set_title("Schedule sample: 0 Off, 1 Morning, 2 Afternoon, 3 Night")
    figure.colorbar(image, ax=axes[3], ticks=[0, 1, 2, 3], label="Shift code")

    audit_scores = ethical_scheduler.audit_system.compliance_history
    axes[4].plot(range(1, len(audit_scores) + 1), audit_scores, marker="o")
    axes[4].set_xlabel("Audit record")
    axes[4].set_ylabel("Weighted indicator score")
    axes[4].set_ylim(0, 1)
    axes[4].set_title("Recorded scores in this session")

    hours_by_employee = {employee.id: 0 for employee in employees}
    for employee_id, _ in ethical_schedule:
        hours_by_employee[employee_id] += 8
    axes[5].hist(list(hours_by_employee.values()), bins=10, color="skyblue", edgecolor="black")
    axes[5].set_xlabel("Assigned hours in the modeled week")
    axes[5].set_ylabel("Number of employees")
    axes[5].set_title("Workload distribution")

    weights = ethical_scheduler.optimization_weights
    axes[6].bar(
        ["Operational efficiency", "Ethical compliance"],
        [weights["operational_efficiency"], weights["ethical_compliance"]],
        color=["royalblue", "mediumseagreen"],
    )
    axes[6].set_ylim(0, 1)
    axes[6].set_ylabel("Weight")
    axes[6].set_title("Greedy ranking weights")
    axes[6].tick_params(axis="x", rotation=25)

    axes[7].bar(["Demand coverage", "Weighted indicator"],
                [sum(ethical_scheduler.coverage.values()) / sum(
                    ethical_scheduler.demand[day][shift.name]
                    for day in days for shift in ethical_scheduler.shifts
                ), overall_score], color=["slateblue", "teal"])
    axes[7].set_ylim(0, 1)
    axes[7].set_ylabel("Value")
    axes[7].set_title("Coverage and indicator score")
    axes[7].tick_params(axis="x", rotation=20)

    figure.suptitle("Scheduling Indicators and Demonstration Data", fontsize=15)
    figure.tight_layout()
    plt.show()


create_ethical_visualizations(
    compliance_scores,
    bias_analysis,
    stakeholder_system,
    ethical_schedule_df,
    employees,
    ethical_schedule,
    ethical_scheduler,
    days,
    overall_score,
)

In [ ]:
def analyze_ethical_benefits(ethical_framework, compliance_scores, overall_score):
    """Summarize measured indicators and distinguish them from external targets."""
    print("\nETHICAL SCHEDULING INDICATOR SUMMARY")
    print(f"Weighted indicator score: {overall_score:.2%}")
    for principle, score in compliance_scores.items():
        print(f"  {principle.replace('_', ' ').title()}: {score:.2%}")

    source_scenario_targets = {
        "Burnout reduction": 35,
        "Staff satisfaction improvement": 40,
        "Retention increase": 25,
        "Regulatory compliance": 100,
    }
    print("\nIllustrative targets stated for the healthcare scenario:")
    for metric, target in source_scenario_targets.items():
        print(f"  {metric}: {target}% (not measured by this model)")

    print("\nInterpretation")
    print("  Schedule data support workload and coverage calculations.")
    print("  Synthetic feedback is a software demonstration, not stakeholder evidence.")
    print("  No causal outcome, regulatory, burnout, satisfaction, or retention claim is tested.")
    return {
        "ethical_metrics": {
            "overall_score": overall_score,
            "compliance_scores": dict(compliance_scores),
        },
        "scenario_targets_not_measured": source_scenario_targets,
    }


ethical_benefits = analyze_ethical_benefits(
    ethical_framework, compliance_scores, overall_score
)

In [ ]:
def demonstrate_ethical_constraints(ethical_framework, ethical_schedule, employees, days):
    """Check implemented schedule limits and identify what the data cannot test."""
    constraints = ethical_framework.constraints
    day_order = {day: index for index, day in enumerate(days)}
    assignments_by_employee = {employee.id: [] for employee in employees}
    for (employee_id, day), shift_name in ethical_schedule.items():
        assignments_by_employee[employee_id].append((day, shift_name))

    consecutive_ok = []
    weekly_hours_ok = []
    days_off_ok = []
    night_limit_ok = []
    distribution = []
    for employee in employees:
        assignments = assignments_by_employee[employee.id]
        assigned_days = sorted(day_order[day] for day, _ in assignments)
        longest_run = 0
        current_run = 0
        previous_day = None
        for day_index in assigned_days:
            current_run = current_run + 1 if previous_day == day_index - 1 else 1
            longest_run = max(longest_run, current_run)
            previous_day = day_index
        consecutive_ok.append(
            longest_run <= constraints["max_consecutive_shifts"]
        )
        weekly_hours = len(assignments) * 8
        weekly_hours_ok.append(weekly_hours <= constraints["max_weekly_hours"])
        days_off_ok.append(
            len(days) - len(set(day for day, _ in assignments))
            >= constraints["min_days_off_per_week"]
        )
        night_count = sum(shift == "Night" for _, shift in assignments)
        weekly_night_limit = constraints["max_night_shifts_per_month"] // 4
        night_limit_ok.append(night_count <= weekly_night_limit)
        distribution.append(weekly_hours)

    mean_hours = float(np.mean(distribution)) if distribution else 0.0
    relative_range = (
        (max(distribution) - min(distribution)) / mean_hours
        if distribution and mean_hours > 0 else 0.0
    )
    compliance = {
        "max_consecutive_shifts": float(np.mean(consecutive_ok)) if employees else 1.0,
        "max_weekly_hours": float(np.mean(weekly_hours_ok)) if employees else 1.0,
        "min_days_off_per_week": float(np.mean(days_off_ok)) if employees else 1.0,
        "weekly_night_limit_proxy": float(np.mean(night_limit_ok)) if employees else 1.0,
        "fair_distribution_threshold_met": float(
            relative_range <= constraints["fair_distribution_threshold"]
        ),
    }

    print("\nCONSTRAINT CHECKS FOR THE GENERATED SCHEDULE")
    for name, rate in compliance.items():
        print(f"  {name.replace('_', ' ').title()}: {rate:.1%}")
    print("  Minimum rest hours: not testable without shift start and end times.")
    print("  Monthly night-shift limit: approximated here by one quarter of the monthly limit.")
    print("  Fair-distribution threshold: reported as an indicator, not enforced by selection.")
    return compliance


constraint_compliance = demonstrate_ethical_constraints(
    ethical_framework, ethical_schedule, employees, days
)

## Summary and interpretation

### What the procedure demonstrates

- A sequential greedy scheduler can rank eligible employees using a weighted operational and workload-related score.
- Availability and selected workload limits are checked as each day-shift demand is processed. Each employee can receive at most one shift per day in the schedule representation.
- Demand coverage, assigned hours, selected constraint checks, and group assignment-rate gaps can be computed from the generated schedule.
- Feedback and audit classes illustrate data structures for recording information; synthetic feedback is not evidence from actual stakeholders.

### How to interpret the indicators

A high value for an individual indicator means only that the corresponding calculation scored well under the definitions used here. The weighted score combines unlike dimensions and should not be treated as a universal ethical measure. In particular, transparency and accountability require meaningful explanations, review processes, and governance—not merely a number or an in-memory log. A small disparity indicator does not establish fairness, and a large one is a prompt for investigation rather than proof of discrimination.

The scheduler is a greedy heuristic. Its choices depend on the order of days and shifts, employee availability, and random ranking scores. It does not search all feasible schedules, guarantee full coverage, or demonstrate global optimality. The maximum-hours, consecutive-day, and night-shift checks are implemented in candidate selection; rest hours cannot be tested because the scenario has no shift times. The fairness threshold is evaluated after scheduling and is not a hard selection constraint.

### Limitations and responsible use

The healthcare scenario's illustrative outcome targets—such as reductions in burnout or increases in satisfaction and retention—are not measured by this program. Evaluating such outcomes would require appropriate baseline and follow-up data, a defensible study design, privacy protections, and stakeholder participation. Before any operational use, decision rules, labor agreements, qualifications, local regulations, accessibility needs, and worker preferences would require careful validation and human oversight.